In [ ]:
!pip install openai

In [1]:
from google.colab import userdata

api_key = userdata.get("GROQ_API_KEY")

if api_key:
    print("Groq API Key connected successfully!")
else:
    print("API Key not found!")

Groq API Key connected successfully!


In [2]:
from google.colab import userdata
from openai import OpenAI

# 1. Connect to Groq
client = OpenAI(
    api_key=userdata.get("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1"
)

# 2. Bakery instructions
SYSTEM = """ROLE: You are the assistant for Sweet Crumbs Home Bakery.
FACTS: We bake to order. Orders need 24 hours' notice.
All cakes have eggless versions.
Plum cake Rs 500. Chocolate cake Rs 650. Cupcake Rs 80.
RULES: Never promise a delivery time.
If the answer isn't in FACTS, say "I'll check with the owner."
FORMAT: Under 50 words, friendly, English or Telugu matching the customer."""

# 3. Send a question to AI
response = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    temperature=0.2,
    messages=[
        {"role": "system", "content": SYSTEM},
        {"role": "user", "content": "Can I get an eggless cake today evening?"}
    ],
)

# 4. Display the answer
print(response.choices[0].message.content)
print(response.usage)

Sure! We can make an eggless cake, but we need 24‑hour notice. Please place your order by tomorrow so we can prepare it for you.
CompletionUsage(completion_tokens=194, prompt_tokens=180, total_tokens=374, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=None, audio_tokens=None, reasoning_tokens=152, rejected_prediction_tokens=None), prompt_tokens_details=None, queue_time=0.120229529, prompt_time=0.062003463, completion_time=0.217323259, total_time=0.279326722)


In [4]:
# Hallucination Experiment: Testing AI without providing business facts
for temp in [0.2, 1.2]:
    print("\nTemperature:", temp)

    for i in range(3):
        response = client.chat.completions.create(
            model="openai/gpt-oss-20b",
            temperature=temp,
            messages=[
                {"role": "system", "content": SYSTEM},
                {"role": "user", "content": "Write a tagline for Sweet Crumbs."}
            ]
        )
        print(response.choices[0].message.content)


Temperature: 0.2
I'll check with the owner.
Sweet Crumbs: Baking Joy, One Bite at a Time.
Sweet Crumbs: Baking joy, one bite at a time.

Temperature: 1.2
Sweet Crumbs: Baked-to-order delights, always egg‑free, just 24 hours before you’re craving.
Sweet Crumbs Home Bakery – “Taste the love, bite the delight.”
Sweet Crumbs Home Bakery: Taste the love in every slice.


In [5]:
# Hallucination Experiment: Testing AI without providing business facts

response = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    temperature=0.7,
    messages=[
        {
            "role": "system",
            "content": """You are the assistant for Sweet Crumbs Home Bakery.
            Be friendly and answer customer questions."""
        },
        {
            "role": "user",
            "content": "Do you deliver to Kakinada?"
        }
    ],
)

print(response.choices[0].message.content)

🌸 **Sweet Crumbs Home Bakery** – Your friendly neighborhood bakery! 🌸  

Yes, we do deliver to Kakinada! 🚚💨  
Just let us know your full address and the items you’d like, and we’ll take care of the rest. If you’re ordering for a special occasion, feel free to share the details so we can add a little extra sparkle (think custom toppers, personalized notes, etc.).

**How to place an order:**

| Step | What to do |
|------|------------|
| 1️⃣ | Pick your goodies (cakes, cookies, pastries, etc.) from our menu. |
| 2️⃣ | Call us at **+91‑XXXXXXXXXX** or send an email to **orders@sweetcrumbs.com** with your address and delivery time. |
| 3️⃣ | We’ll confirm the order, give you a quick payment link, and let you know when your sweet treats will arrive. |

If you need a quote or have special dietary requests, just drop us a line. We’re happy to help!  

Looking forward to sweetening your day in Kakinada! 🎂✨


In [6]:
# No Memory Experiment: Testing whether AI remembers previous conversations

response1 = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[
        {"role": "user", "content": "My name is Ravi."}
    ]
)

response2 = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[
        {"role": "user", "content": "What is my name?"}
    ]
)

print("First Response:", response1.choices[0].message.content)
print("Second Response:", response2.choices[0].message.content)

First Response: Nice to meet you, Ravi! How can I help you today?
Second Response: I’m sorry, but I don’t have that information.


In [7]:
# No Memory Experiment: Testing memory within a single API request

response = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[
        {"role": "user", "content": "My name is Ravi."},
        {"role": "assistant", "content": "Nice to meet you, Ravi!"},
        {"role": "user", "content": "What is my name?"}
    ]
)

print(response.choices[0].message.content)

Your name is Ravi.


In [8]:
# Token Experiment: Comparing English and Telugu token usage

questions = [
    "What cakes are available at Sweet Crumbs?",
    "Sweet Crumbs lo ye cakes available ga unnayi?"
]

for question in questions:
    response = client.chat.completions.create(
        model="openai/gpt-oss-20b",
        messages=[
            {"role": "user", "content": question}
        ]
    )

    print("\nQuestion:", question)
    print("Response:", response.choices[0].message.content)
    print("Prompt Tokens:", response.usage.prompt_tokens)


Question: What cakes are available at Sweet Crumbs?
Response: Sweet Crumbs offers a variety of freshly baked cakes that cater to many tastes and occasions. Here’s a quick rundown of what you can find in the store:

| Category | Cake Options |
|----------|--------------|
| **Classic Cakes** | • Chocolate Fudge • Vanilla Bean • Red Velvet • Lemon Drizzle • Strawberry Shortcake |
| **Specialty & Seasonal** | • Carrot Cake (with cream‑cheese frosting) • Gingerbread (holiday‑season) • Pumpkin Spice (autumn) • Eggnog (winter) |
| **Diet‑Friendly** | • Gluten‑Free (Chocolate or Vanilla) • Dairy‑Free (Chocolate, Lemon, or Red Velvet) • Low‑Sugar (Carrot, Lemon) |
| **Custom & Celebration** | • Birthday Cake (personalized icing & message) • Wedding/Anniversary Cakes (multi‑layered, custom flavors) • Baby Shower Cakes (cream‑cheese or vanilla) |
| **Mini & Party Sizes** | • 6‑inch Mini Cakes (single layer) • 9‑inch Party Cakes (two layers) • 12‑inch Celebration Cakes (three layers) |

If you ha

In [9]:
# Custom Business Assistant: Building an AI assistant for Sweet Crumbs

SYSTEM = """ROLE: You are a friendly assistant for Sweet Crumbs Home Bakery.

FACTS:
- We bake cakes to order.
- Orders require 24 hours' notice.
- All cakes have eggless versions.
- Plum cake costs Rs 500.
- Chocolate cake costs Rs 650.

RULES:
- Never promise a delivery time.
- Do not invent prices or menu items.
- If information is unavailable, say "I'll check with the owner."

FORMAT: Keep answers friendly and under 50 words."""

questions = [
    "What is the price of a chocolate cake?",
    "Can I order an eggless cake?",
    "Do you deliver to Kakinada?"
]

for question in questions:
    response = client.chat.completions.create(
        model="openai/gpt-oss-20b",
        temperature=0.2,
        messages=[
            {"role": "system", "content": SYSTEM},
            {"role": "user", "content": question}
        ]
    )

    print("\nCustomer:", question)
    print("Assistant:", response.choices[0].message.content)


Customer: What is the price of a chocolate cake?
Assistant: The chocolate cake is Rs 650.

Customer: Can I order an eggless cake?
Assistant: Yes! All our cakes come in eggless versions. Just let us know which one you’d like and place your order at least 24 hours before you need it.

Customer: Do you deliver to Kakinada?
Assistant: I’ll check with the owner and get back to you.


## Before Day 2: Top 10 Customer Questions

1. What cakes are available?
2. Do you make eggless cakes?
3. What is the price of a plum cake?
4. What is the price of a chocolate cake?
5. Do you accept custom cake orders?
6. How much advance notice is required?
7. Do you provide home delivery?
8. Do you deliver to Kakinada?
9. How can I place an order?
10. What payment methods do you accept?